# Introduzione alla Modellazione Semantica con dbt

Questo notebook accompagna la lezione sulla **modellazione semantica** usando dbt e il database AdventureWorks.

## Obiettivi della Lezione

1. Comprendere i problemi delle query SQL dirette
2. Il problema del fanout
3. Il problema dei campi non allineati
4. Come dbt risolve questi problemi

## Cos'e la Modellazione Semantica?

La modellazione semantica crea un **livello di astrazione** tra i dati grezzi e le query business.

Invece di scrivere query SQL complesse ogni volta, gli analisti usano **modelli predefiniti** che:
- Applicano automaticamente i filtri corretti
- Usano le aggregazioni giuste (DISTINCT dove serve)
- Calcolano le metriche dai dati dettagliati
- Sono **corretti by design**

## 1. Esplorazione dei Dati Grezzi

I dati sono memorizzati in tabelle nel database DuckDB.

DuckDB e un database embedded (non richiede installazione), perfetto per demo e test.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect('adventureworks-dbt-course/adventureworks/data/adventureworks.duckdb')

print('Tabelle disponibili:')
tables = con.execute('SHOW TABLES').fetchall()
for t in tables:
    print(f'  - {t[0]}')

### 1.1 Tabella Clienti (customers)

La tabella customers contiene i dati anagrafici dei clienti. Include anche la citta per analisi geografiche.

In [ ]:
customers = con.execute('SELECT * FROM customers').df()
print('TABELLA CLIENTI:')
print(customers.to_string(index=False))

### 1.2 Tabella Ordini (orders)

La tabella orders contiene le testate degli ordini. Il campo **status** e importante:
- 1 = Pending
- 5 = Shipped
- 6 = Cancelled

**Regola**: considerare solo gli ordini spediti (status=5)!

In [ ]:
orders = con.execute('SELECT * FROM orders').df()
print('TABELLA ORDINI:')
print(orders.to_string(index=False))

### 1.3 Tabella Righe Ordine (order_lines)

Contiene i singoli prodotti per ogni ordine:
- **quantity**: quantita ordinata
- **unit_price**: prezzo unitario
- **line_total**: totale della riga
- **discount**: percentuale di sconto!

**Nota**: Gli sconti sono applicati a livello di riga!

In [ ]:
order_lines = con.execute('SELECT * FROM order_lines').df()
print('TABELLA RIGHE ORDINE:')
print(order_lines.to_string(index=False))

---

## 2. IL PROBLEMA DEL FANOUT

Il **fanout** e uno dei problemi piu comuni nelle query SQL.

**Quando succede**: Quando fai un JOIN tra tabelle e poi un'aggregazione senza le dovute accortezze.

**Esempio**: Un cliente fa 1 ordine con 3 prodotti. COUNT(order_id) restituisce 3, non 1!

### Esempio 1: Query Sbagliata (Fanout)

Questa e la query che scriverebbe un analista inexperto:

In [ ]:
query_sbagliata = '''
SELECT 
    c.customer_id,
    c.first_name || " " || c.last_name AS customer_name,
    COUNT(ol.order_id) AS order_count
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_lines ol ON o.order_id = ol.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
'''

result_sbagliato = con.execute(query_sbagliata).df()
print("QUERY SBAGLIATA (senza DISTINCT):")
print("Conta le righe, non gli ordini!")
print(result_sbagliato.to_string(index=False))

### Come risolvere il fanout

Usa **COUNT(DISTINCT ...)** invece di COUNT per contare valori unici.

In [ ]:
query_corretta = '''
SELECT 
    c.customer_id,
    c.first_name || " " || c.last_name AS customer_name,
    COUNT(DISTINCT o.order_id) AS order_count
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_lines ol ON o.order_id = ol.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
'''

result_corretto = con.execute(query_corretta).df()
print("QUERY CORRETTA (con DISTINCT):")
print(result_corretto.to_string(index=False))

In [ ]:
comparison = result_sbagliato.merge(result_corretto, on="customer_id", suffixes=("_sbagliato", "_corretto"))
comparison["differenza"] = comparison["order_count_sbagliato"] - comparison["order_count_corretto"]

print("="*60)
print("DIFFERENZA TRA LE DUE QUERY:")
print("="*60)
print(" customer_name  | sbagliato | corretto | differenza")
print("-" * 55)
for _, row in comparison.iterrows():
    print(f"{row['customer_name']:14} | {row['order_count_sbagliato']:8} | {row['order_count_corretto']:7} | {row['differenza']:9}")

print("La differenza corrisponde al numero di prodotti per ordine!")

---

## 3. Problema 2: Campi Non Allineati

Un altro problema comune: usare il campo totale dellintestazione invece di calcolare dalle righe ordine.

In [ ]:
query_header = '''SELECT    c.customer_id,    c.first_name || ' ' || c.last_name AS customer_name,    SUM(o.total) AS lifetime_valueFROM customers cJOIN orders o ON c.customer_id = o.customer_idWHERE o.status = 5GROUP BY c.customer_id, c.first_name, c.last_name'''result_header = con.execute(query_header).df()print("QUERY SBAGLIATA (usa orders.total):")print(result_header.to_string(index=False))

In [ ]:
query_lines = '''SELECT    c.customer_id,    c.first_name || ' ' || c.last_name AS customer_name,    SUM(ol.line_total * (1 - ol.discount)) AS lifetime_valueFROM customers cJOIN orders o ON c.customer_id = o.customer_idJOIN order_lines ol ON o.order_id = ol.order_idWHERE o.status = 5GROUP BY c.customer_id, c.first_name, c.last_name'''result_lines = con.execute(query_lines).df()print("QUERY CORRETTA (calcola da order_lines):")print(result_lines.to_string(index=False))

In [ ]:
comp = result_header.merge(result_lines, on="customer_id", suffixes=("_header", "_lines"))comp["differenza"] = comp["lifetime_value_header"] - comp["lifetime_value_lines"]print("="*60)print("DIFFERENZA:")print("="*60)for _, row in comp.iterrows():    print(f"{row["customer_name"]:14} | {row["lifetime_value_header"]:10.2f} | {row["lifetime_value_lines"]:10.2f} | {row["differenza"]:9.2f}")

---

## 4. Come dbt Risolve Questi Problemi

Ora vediamo come i modelli dbt risolvono questi problemi automaticamente.

In [ ]:
print("FACT TABLES CREATE DA DBT:")print()fct_customers = con.execute("SELECT customer_id, first_name, last_name, city, order_count, lifetime_gross_value, lifetime_net_value FROM fct_customers").df()print("fct_customers:")print(fct_customers.to_string(index=False))

In [ ]:
fct_products = con.execute("SELECT product_id, product_name, category_name, orders_with_product, units_sold, gross_revenue, net_revenue FROM fct_products").df()print()print("fct_products:")print(fct_products.to_string(index=False))

In [ ]:
fct_orders = con.execute("SELECT order_id, order_date, customer_name, order_header_total, gross_revenue, net_revenue FROM fct_orders").df()print()print("fct_orders:")print(fct_orders.to_string(index=False))

### Cosa fanno i modelli dbt:

1. fct_customers: COUNT(DISTINCT order_id) + calcola da order_lines
2. fct_products: COUNT(DISTINCT) + revenue con sconti
3. fct_orders: gross vs net revenue

---

## Conclusione

Con dbt, basta:
```sql
SELECT * FROM fct_customers
```

Risultati corretti by design!